# Evaluate Best Models from Multiple Runs

Simple notebook to load and compare best models from different checkpoint runs.

## 1. Setup and Load Models

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import distributions as dist
from torch.utils.data import Dataset, DataLoader

from synthetic_peptides_dataset import SyntheticPeptidesDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

def load_symbols_from_notebook(nb_path, symbols):
    nb = json.loads(Path(nb_path).read_text())
    hits = 0
    for cell in nb["cells"]:
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        if any((f"class {s}" in src) or (f"def {s}(" in src) for s in symbols):
            exec(src, globals(), globals())
            hits += 1
    print(f"Loaded {hits} definition cells from {Path(nb_path).name}")

load_symbols_from_notebook(
    "train_synthetic_peptides.ipynb",
    ["knn", "get_graph_feature", "PointNetPlusPlusEncoder", "AdaptivePointDecoder", "PointNetPlusPlusVAE"],
)
load_symbols_from_notebook(
    "convocc_net_peptides.ipynb",
    [
        "normalize_3d_coordinate", "ResnetBlockFC", "conv3d", "SingleConv", "DoubleConv",
        "Encoder", "UNet3DEncoder", "LocalDecoder", "ConvolutionalOccupancyNetwork",
    ],
)

print("Model definitions loaded from notebooks")

Device: cuda
Model defined with Lightweight AdaptivePointDecoder
  - Reduced parameters (removed BatchNorm, narrower layers)
  - Volumetric generation (not surface-only)
  - Learned seed points + refinement
  - Supports variable point counts
Loaded 1 definition cells from train_synthetic_peptides.ipynb


FileNotFoundError: [Errno 2] No such file or directory: 'convocc_net_peptides.ipynb'

In [ ]:
# Load target runs for both families
checkpoint_dir = Path("checkpoints")
TARGET_SIZES = [300, 1000, 2000]


def ckpt_path(run_name):
    d = checkpoint_dir / run_name
    p = d / "best.pth"
    return p if p.exists() else d / "latest.pth"


def build_model(run_name, cfg):
    if run_name.startswith("simulated_data"):
        model = PointNetPlusPlusVAE(
            latent_dim=cfg.get("latent_dim", 128),
            num_points=cfg.get("n_points", 2000),
        )
        kind = "pointcloud"
    else:
        model = ConvolutionalOccupancyNetwork(
            decoder=LocalDecoder(dim=3, c_dim=64, hidden_size=256, n_blocks=5, padding=0.1),
            encoder=UNet3DEncoder(in_channels=1, f_maps=16, num_levels=3, layer_order="gcr", num_groups=8),
            device=device,
        )
        kind = "voxel"
    return model.to(device), kind


run_names = []
for size in TARGET_SIZES:
    run_names.extend(
        [
            f"convocc_peptides_{size}",
            f"convocc_peptides_focal_{size}",
            f"simulated_data_{size}",
        ]
    )

models = {}
missing = []
for run_name in run_names:
    d = checkpoint_dir / run_name
    if not d.exists():
        missing.append(run_name)
        continue

    path = ckpt_path(run_name)
    if not path.exists():
        missing.append(run_name)
        continue

    ckpt = torch.load(path, map_location=device)
    cfg = ckpt.get("config", {})
    model, kind = build_model(run_name, cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    models[run_name] = {
        "model": model,
        "kind": kind,
        "config": cfg,
        "epoch": ckpt.get("epoch", "N/A"),
    }
    print(f"Loaded {run_name} ({kind}) from {path}")

if missing:
    print(f"Missing runs/checkpoints: {missing}")

print(f"\nLoaded models: {list(models.keys())}")

/tmp/ipykernel_102716/4150581798.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=device)


Loaded convocc_peptides_300 (voxel) from checkpoints/convocc_peptides_300/best.pth
Loaded simulated_data_300 (pointcloud) from checkpoints/simulated_data_300/best.pth
Loaded convocc_peptides_1000 (voxel) from checkpoints/convocc_peptides_1000/best.pth
Loaded simulated_data_1000 (pointcloud) from checkpoints/simulated_data_1000/best.pth
Loaded convocc_peptides_2000 (voxel) from checkpoints/convocc_peptides_2000/best.pth
Loaded simulated_data_2000 (pointcloud) from checkpoints/simulated_data_2000/best.pth

Loaded models: ['convocc_peptides_300', 'simulated_data_300', 'convocc_peptides_1000', 'simulated_data_1000', 'convocc_peptides_2000', 'simulated_data_2000']


## 2. Compute Metrics on Test Set

In [5]:
# Datasets + representation converters
class PointCloudDataset(Dataset):
    def __init__(self, data_path, n_points=2000, num_files=None):
        self.base = SyntheticPeptidesDataset(
            data_path=data_path,
            num_files=num_files,
            target_num_points=None,
            normalize=True,
            return_peptide_ids=False,
        )
        self.n_points = n_points

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        sample = self.base[idx]
        pts = sample["points"].cpu().numpy()  # [N, 3]
        choose = np.random.choice(len(pts), self.n_points, replace=len(pts) < self.n_points)
        pts = pts[choose]
        return torch.tensor(pts.T, dtype=torch.float32), sample["label"]  # [3, n_points], structure


class VoxelDataset(Dataset):
    def __init__(self, data_path, grid_size=64, num_files=None):
        self.base = SyntheticPeptidesDataset(
            data_path=data_path,
            num_files=num_files,
            target_num_points=None,
            normalize=True,
            return_peptide_ids=False,
        )
        self.grid_size = grid_size

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        sample = self.base[idx]
        pts = sample["points"].cpu().numpy()  # [N, 3]
        voxel = torch.tensor(points_to_voxels(pts, self.grid_size), dtype=torch.float32).unsqueeze(0)
        return voxel, sample["label"]


def points_to_voxels(points, grid_size=64):
    vox = np.zeros((grid_size, grid_size, grid_size), dtype=np.float32)
    idx = np.clip(((points + 0.5) * (grid_size - 1)).astype(int), 0, grid_size - 1)
    vox[idx[:, 0], idx[:, 1], idx[:, 2]] = 1.0
    return vox


def voxels_to_points(voxels, threshold=0.5):
    # voxels: [D,H,W] numpy/torch in [0,1]
    if isinstance(voxels, torch.Tensor):
        vox = voxels.detach().cpu().numpy()
    else:
        vox = voxels
    idx = np.argwhere(vox > threshold)
    if len(idx) == 0:
        return np.zeros((0, 3), dtype=np.float32)
    pts = idx.astype(np.float32) / (np.array(vox.shape, dtype=np.float32) - 1.0) - 0.5
    return pts


print("Dataset classes ready (with structure labels + pointcloud<->voxel converters)")

Dataset classes ready (with structure labels + pointcloud<->voxel converters)


In [8]:
# Metrics

def chamfer_distance(p1, p2):
    if p1.size(1) == 3:
        p1 = p1.transpose(1, 2)
    if p2.size(1) == 3:
        p2 = p2.transpose(1, 2)
    d = torch.cdist(p1, p2, p=2)
    return torch.mean(torch.min(d, dim=2)[0]) + torch.mean(torch.min(d, dim=1)[0])


def safe_chamfer_from_np(pred_pts, gt_pts, device):
    # Avoid NaN when one side is empty after voxel thresholding.
    if len(pred_pts) == 0 and len(gt_pts) == 0:
        return 0.0
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return 1.0
    p1 = torch.tensor(pred_pts.T, dtype=torch.float32, device=device).unsqueeze(0)
    p2 = torch.tensor(gt_pts.T, dtype=torch.float32, device=device).unsqueeze(0)
    return float(chamfer_distance(p1, p2).item())


def earth_moving_distance_from_np(pred_pts, gt_pts, max_points=512):
    # Uses Hungarian matching when scipy is available; otherwise falls back to symmetric NN distance.
    if len(pred_pts) == 0 and len(gt_pts) == 0:
        return 0.0
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return 1.0

    n = min(len(pred_pts), len(gt_pts), max_points)
    pred = pred_pts[np.random.choice(len(pred_pts), n, replace=False)].astype(np.float32)
    gt = gt_pts[np.random.choice(len(gt_pts), n, replace=False)].astype(np.float32)

    try:
        from scipy.optimize import linear_sum_assignment
        cost = np.linalg.norm(pred[:, None, :] - gt[None, :, :], axis=2)
        r, c = linear_sum_assignment(cost)
        return float(cost[r, c].mean())
    except Exception:
        pred_t = torch.tensor(pred, dtype=torch.float32).unsqueeze(0)
        gt_t = torch.tensor(gt, dtype=torch.float32).unsqueeze(0)
        d = torch.cdist(pred_t, gt_t, p=2).squeeze(0)
        return float(0.5 * (d.min(dim=1)[0].mean() + d.min(dim=0)[0].mean()).item())


def coverage_from_np(pred_pts, gt_pts, radius=0.02):
    # Fraction of GT points covered by prediction within a distance threshold.
    if len(gt_pts) == 0:
        return 1.0
    if len(pred_pts) == 0:
        return 0.0
    pred_t = torch.tensor(pred_pts, dtype=torch.float32).unsqueeze(0)
    gt_t = torch.tensor(gt_pts, dtype=torch.float32).unsqueeze(0)
    d = torch.cdist(gt_t, pred_t, p=2).squeeze(0)
    return float((d.min(dim=1)[0] <= radius).float().mean().item())


def jsd_from_np(pred_pts, gt_pts, bins=28, eps=1e-8):
    # Jensen-Shannon divergence between normalized 3D occupancy histograms.
    edges = np.linspace(-0.5, 0.5, bins + 1, dtype=np.float32)
    h_pred, _ = np.histogramdd(pred_pts, bins=(edges, edges, edges))
    h_gt, _ = np.histogramdd(gt_pts, bins=(edges, edges, edges))
    p = h_pred.reshape(-1).astype(np.float64) + eps
    q = h_gt.reshape(-1).astype(np.float64) + eps
    p /= p.sum()
    q /= q.sum()
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))
    return float(0.5 * (kl_pm + kl_qm))


def occupancy_auc_roc(pred_scores, gt_occ):
    # Handles edge cases where only one class is present in GT.
    pred = pred_scores.detach().reshape(-1).float().cpu()
    gt = (gt_occ.detach().reshape(-1) > 0.5).float().cpu()

    n_pos = int(gt.sum().item())
    n = gt.numel()
    n_neg = n - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    order = torch.argsort(pred, descending=True)
    gt_sorted = gt[order]
    tp = torch.cumsum(gt_sorted, dim=0)
    fp = torch.cumsum(1.0 - gt_sorted, dim=0)
    tpr = tp / n_pos
    fpr = fp / n_neg

    tpr = torch.cat([torch.tensor([0.0]), tpr, torch.tensor([1.0])])
    fpr = torch.cat([torch.tensor([0.0]), fpr, torch.tensor([1.0])])
    auc = torch.trapz(tpr, fpr).item()
    return float(max(0.0, min(1.0, auc)))


def make_grid_query_points(grid_size, device):
    lin = torch.linspace(-0.5, 0.5, grid_size, device=device)
    xyz = torch.stack(torch.meshgrid(lin, lin, lin, indexing="ij"), dim=-1)
    return xyz.reshape(-1, 3)


print("Metrics ready: occupancy_bce, accuracy, occupancy_auc_roc, chamfer_distance, earth_moving_distance, coverage, jsd")

Metrics ready: occupancy_bce, accuracy, occupancy_auc_roc, chamfer_distance, earth_moving_distance, coverage, jsd


In [9]:
# Evaluate both models with a common metric set + per-structure tables
results = []
structure_rows = {}
NUM_FILES_BY_SIZE = {300: 64, 1000: 213, 2000: 426}


def num_files_for_run(run_name):
    try:
        size = int(run_name.rsplit("_", 1)[-1])
        return NUM_FILES_BY_SIZE.get(size, None)
    except Exception:
        return None


for run_name, m in models.items():
    model, kind, cfg = m["model"], m["kind"], m["config"]
    print(f"\nEvaluating {run_name} ({kind})...")

    if kind == "pointcloud":
        n_points = cfg.get("n_points", 2000)
        num_files = num_files_for_run(run_name)
        grid_size = cfg.get("grid_size", 64)

        ds = PointCloudDataset("./data/synthetic_peptides_split/test", n_points=n_points, num_files=num_files)
        loader = DataLoader(ds, batch_size=16, shuffle=False)

        bces, accs, aucs, cds, emds, covs, jsds = [], [], [], [], [], [], []
        per_struct = {}
        with torch.no_grad():
            for batch, labels in loader:
                batch = batch.to(device)  # [B,3,N]
                recon, _, _ = model(batch)

                for i, label in enumerate(labels):
                    gt_pts = batch[i].transpose(0, 1).cpu().numpy()  # [N,3]
                    pr_pts = recon[i].transpose(0, 1).cpu().numpy()  # [N,3]

                    gt_vox = torch.tensor(points_to_voxels(gt_pts, grid_size), dtype=torch.float32, device=device)
                    pr_vox = torch.tensor(points_to_voxels(pr_pts, grid_size), dtype=torch.float32, device=device)

                    bce = F.binary_cross_entropy(pr_vox.clamp(1e-4, 1 - 1e-4), gt_vox).item()
                    acc = (pr_vox.round() == gt_vox).float().mean().item()
                    auc = occupancy_auc_roc(pr_vox, gt_vox)
                    cd = chamfer_distance(recon[i:i+1], batch[i:i+1]).item()
                    emd = earth_moving_distance_from_np(pr_pts, gt_pts)
                    cov = coverage_from_np(pr_pts, gt_pts)
                    jsd = jsd_from_np(pr_pts, gt_pts)

                    bces.append(bce)
                    accs.append(acc)
                    aucs.append(auc)
                    cds.append(cd)
                    emds.append(emd)
                    covs.append(cov)
                    jsds.append(jsd)

                    s = per_struct.setdefault(label, {"bce": [], "acc": [], "auc_roc": [], "cd": [], "emd": [], "coverage": [], "jsd": []})
                    s["bce"].append(bce)
                    s["acc"].append(acc)
                    s["auc_roc"].append(auc)
                    s["cd"].append(cd)
                    s["emd"].append(emd)
                    s["coverage"].append(cov)
                    s["jsd"].append(jsd)

        results.append({
            "run": run_name,
            "type": kind,
            "epoch": m["epoch"],
            "structure": "ALL",
            "num_eval_files": len(ds),
            "occupancy_bce": float(np.nanmean(bces)),
            "occupancy_accuracy": float(np.nanmean(accs)),
            "occupancy_auc_roc": float(np.nanmean(aucs)),
            "chamfer_distance": float(np.nanmean(cds)),
            "earth_moving_distance": float(np.nanmean(emds)),
            "coverage": float(np.nanmean(covs)),
            "jsd": float(np.nanmean(jsds)),
        })
        print(
            f"  Files: {len(ds)}, BCE: {np.nanmean(bces):.6f}, Acc: {np.nanmean(accs):.4f}, "
            f"AUC: {np.nanmean(aucs):.4f}, Chamfer: {np.nanmean(cds):.6f}, "
            f"EMD: {np.nanmean(emds):.6f}, Cov: {np.nanmean(covs):.4f}, JSD: {np.nanmean(jsds):.6f}"
        )

        for st, vals in per_struct.items():
            row = {
                "run": run_name,
                "type": kind,
                "epoch": m["epoch"],
                "structure": st,
                "num_eval_files": len(vals["bce"]),
                "occupancy_bce": float(np.nanmean(vals["bce"])),
                "occupancy_accuracy": float(np.nanmean(vals["acc"])),
                "occupancy_auc_roc": float(np.nanmean(vals["auc_roc"])),
                "chamfer_distance": float(np.nanmean(vals["cd"])),
                "earth_moving_distance": float(np.nanmean(vals["emd"])),
                "coverage": float(np.nanmean(vals["coverage"])),
                "jsd": float(np.nanmean(vals["jsd"])),
            }
            structure_rows.setdefault(st, []).append(row)

    else:
        grid_size = cfg.get("grid_size", 64)
        num_files = num_files_for_run(run_name)
        ds = VoxelDataset("./data/synthetic_peptides_split/test", grid_size=grid_size, num_files=num_files)
        loader = DataLoader(ds, batch_size=1, shuffle=False)

        grid_q = make_grid_query_points(grid_size, device).unsqueeze(0)  # [1, G^3, 3]
        bces, accs, aucs, cds, emds, covs, jsds = [], [], [], [], [], [], []
        per_struct = {}

        with torch.no_grad():
            for voxel, label in loader:
                st = label[0] if isinstance(label, (list, tuple)) else label
                voxel = voxel.to(device)  # [1,1,G,G,G]
                logits = model(grid_q, voxel).logits.view(-1)  # [G^3]
                probs = torch.sigmoid(logits)

                pred_vox = probs.view(grid_size, grid_size, grid_size)
                gt_vox = voxel[0, 0]

                bce = F.binary_cross_entropy_with_logits(logits, gt_vox.reshape(-1)).item()
                acc = ((pred_vox > 0.5) == (gt_vox > 0.5)).float().mean().item()
                auc = occupancy_auc_roc(pred_vox, gt_vox)

                pred_pts = voxels_to_points(pred_vox, threshold=0.5)
                gt_pts = voxels_to_points(gt_vox, threshold=0.5)
                cd = safe_chamfer_from_np(pred_pts, gt_pts, device)
                emd = earth_moving_distance_from_np(pred_pts, gt_pts)
                cov = coverage_from_np(pred_pts, gt_pts)
                jsd = jsd_from_np(pred_pts, gt_pts)

                bces.append(bce)
                accs.append(acc)
                aucs.append(auc)
                cds.append(cd)
                emds.append(emd)
                covs.append(cov)
                jsds.append(jsd)

                s = per_struct.setdefault(st, {"bce": [], "acc": [], "auc_roc": [], "cd": [], "emd": [], "coverage": [], "jsd": []})
                s["bce"].append(bce)
                s["acc"].append(acc)
                s["auc_roc"].append(auc)
                s["cd"].append(cd)
                s["emd"].append(emd)
                s["coverage"].append(cov)
                s["jsd"].append(jsd)

        results.append({
            "run": run_name,
            "type": kind,
            "epoch": m["epoch"],
            "structure": "ALL",
            "num_eval_files": len(ds),
            "occupancy_bce": float(np.nanmean(bces)),
            "occupancy_accuracy": float(np.nanmean(accs)),
            "occupancy_auc_roc": float(np.nanmean(aucs)),
            "chamfer_distance": float(np.nanmean(cds)),
            "earth_moving_distance": float(np.nanmean(emds)),
            "coverage": float(np.nanmean(covs)),
            "jsd": float(np.nanmean(jsds)),
        })
        print(
            f"  Files: {len(ds)}, BCE: {np.nanmean(bces):.6f}, Acc: {np.nanmean(accs):.4f}, "
            f"AUC: {np.nanmean(aucs):.4f}, Chamfer: {np.nanmean(cds):.6f}, "
            f"EMD: {np.nanmean(emds):.6f}, Cov: {np.nanmean(covs):.4f}, JSD: {np.nanmean(jsds):.6f}"
        )

        for st, vals in per_struct.items():
            row = {
                "run": run_name,
                "type": kind,
                "epoch": m["epoch"],
                "structure": st,
                "num_eval_files": len(vals["bce"]),
                "occupancy_bce": float(np.nanmean(vals["bce"])),
                "occupancy_accuracy": float(np.nanmean(vals["acc"])),
                "occupancy_auc_roc": float(np.nanmean(vals["auc_roc"])),
                "chamfer_distance": float(np.nanmean(vals["cd"])),
                "earth_moving_distance": float(np.nanmean(vals["emd"])),
                "coverage": float(np.nanmean(vals["coverage"])),
                "jsd": float(np.nanmean(vals["jsd"])),
            }
            structure_rows.setdefault(st, []).append(row)

results_df = pd.DataFrame(results)
print("\n" + "=" * 80)
print("Overall (ALL structures)")
print(results_df.to_string(index=False))
print("=" * 80)

structure_tables = {st: pd.DataFrame(rows) for st, rows in sorted(structure_rows.items())}
for st, df in structure_tables.items():
    print("\n" + "-" * 80)
    print(f"Structure: {st}")
    print(df.to_string(index=False))
    print("-" * 80)

NameError: name 'models' is not defined

## 3. Compare Results Across Runs

In [42]:
# Compact summary
if not results_df.empty:
    print("\nBest by representation:")
    point_rows = results_df[results_df["type"] == "pointcloud"]
    voxel_rows = results_df[results_df["type"] == "voxel"]

    if not point_rows.empty:
        i = point_rows["chamfer_distance"].idxmin()
        print(
            f"Point cloud best: {results_df.loc[i, 'run']} "
            f"(BCE={results_df.loc[i, 'occupancy_bce']:.6f}, "
            f"Acc={results_df.loc[i, 'occupancy_accuracy']:.4f}, "
            f"Chamfer={results_df.loc[i, 'chamfer_distance']:.6f})"
        )

    if not voxel_rows.empty:
        i = voxel_rows["occupancy_bce"].idxmin()
        print(
            f"Voxel best: {results_df.loc[i, 'run']} "
            f"(BCE={results_df.loc[i, 'occupancy_bce']:.6f}, "
            f"Acc={results_df.loc[i, 'occupancy_accuracy']:.4f}, "
            f"Chamfer={results_df.loc[i, 'chamfer_distance']:.6f})"
        )


Best by representation:
Point cloud best: simulated_data_2000 (BCE=0.078335, Acc=0.9915, Chamfer=0.072528)
Voxel best: convocc_peptides_1000 (BCE=0.072439, Acc=0.9699, Chamfer=0.046919)


## 4. Visualize Best and Worst Cases

In [ ]:
print("Visualization skipped: the two models use different output representations (point cloud vs occupancy field).")
print("Use model-specific visualization notebooks for qualitative comparison.")